In [1]:
# =============================
# Libraries Import
# =============================
import tensorflow as tf
import mlflow
import mlflow.tensorflow
import matplotlib.pyplot as plt
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D
import numpy as np

2026-01-07 03:31:29.312657: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-07 03:31:30.378399: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-07 03:31:32.652726: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# =============================
# MLflow setup
# =============================
TRACKING_URL = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "NAIRA CLASSIFICATION"

mlflow.set_tracking_uri(TRACKING_URL)
mlflow.set_experiment(EXPERIMENT_NAME)

# Enable autologging
mlflow.tensorflow.autolog()


2026/01/07 03:32:08 INFO mlflow.tracking.fluent: Experiment with name 'NAIRA CLASSIFICATION' does not exist. Creating a new experiment.


In [3]:
# =============================
# Dataset paths
# =============================
train_dir = "NAIRA DATASET/Train"
val_dir = "NAIRA DATASET/val"
test_dir = "NAIRA DATASET/Test"


In [4]:
# =============================
# Data generators
# =============================
data_aug = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    width_shift_range=0.2,
    height_shift_range=0.2,
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    shear_range=0.2,
    zoom_range=0.2
)

data = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_dataset = data_aug.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32
)

val_dataset = data.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32
)
test_dataset = data.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32
)


Found 1736 images belonging to 8 classes.
Found 216 images belonging to 8 classes.
Found 90 images belonging to 8 classes.


In [5]:
# =============================
# Model definition
# =============================
img_size = (224, 224)
img_shape = img_size + (3,)

mobilenet = tf.keras.applications.MobileNetV2(
    input_shape=img_shape,
    include_top=False,
    weights="imagenet"
)

mobilenet.trainable = False  # Transfer learning

model = Sequential([
    mobilenet,
    GlobalAveragePooling2D(),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(8, activation="softmax")
])

model.compile(
    optimizer="Adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

2026-01-07 03:32:17.647776: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [7]:
# =============================
# MLflow-tracked training
# =============================
with mlflow.start_run(run_name="mobilenetv2-transfer-learning"):
    model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=5
    )


2026/01/07 03:33:35 WARNING mlflow.tensorflow: Unrecognized dataset type <class 'keras.src.legacy.preprocessing.image.DirectoryIterator'>. Dataset logging skipped.
2026/01/07 03:33:35 WARNING mlflow.tensorflow: Unrecognized dataset type <class 'keras.src.legacy.preprocessing.image.DirectoryIterator'>. Dataset logging skipped.


Epoch 1/5
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 921ms/step - accuracy: 0.6375 - loss: 1.0582

55/55 ━━━━━━━━━━━━━━━━━━━━ 55s 996ms/step - accuracy: 0.7270 - loss: 0.7986 - val_accuracy: 0.7444 - val_loss: 0.6723
Epoch 2/5
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 879ms/step - accuracy: 0.8528 - loss: 0.4137

55/55 ━━━━━━━━━━━━━━━━━━━━ 52s 934ms/step - accuracy: 0.8623 - loss: 0.3798 - val_accuracy: 0.8556 - val_loss: 0.4927
Epoch 3/5
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 905ms/step - accuracy: 0.8912 - loss: 0.3305

55/55 ━━━━━━━━━━━━━━━━━━━━ 53s 952ms/step - accuracy: 0.8975 - loss: 0.3100 - val_accuracy: 0.8444 - val_loss: 0.4407
Epoch 4/5
55/55 ━━━━━━━━━━━━━━━━━━━━ 49s 889ms/step - accuracy: 0.9165 - loss: 0.2414 - val_accuracy: 0.7444 - val_loss: 0.5635
Epoch 5/5
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 870ms/step - accuracy: 0.9162 - loss: 0.2331

55/55 ━━━━━━━━━━━━━━━━━━━━ 51s 920ms/step - accuracy: 0.9130 - loss: 0.2428 - val_accuracy: 0.8111 - val_loss: 0.4393


2026/01/07 03:37:56 WARNING mlflow.tensorflow: Failed to infer model signature: could not sample data to infer model signature: '>=' not supported between instances of 'slice' and 'int'
2026/01/07 03:37:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/07 03:37:56 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/01/07 03:38:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run mobilenetv2-transfer-learning at: http://127.0.0.1:5000/#/experiments/671147824598811407/runs/e44f2f8fec414e5dacba0b940dda5db3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/671147824598811407
